# MINGLE · Qwen EXP-004 시연 서버 (Colab)

밍글 백엔드(`QwenProvider`)가 호출하는 추론 API를 Colab에서 띄우고 ngrok으로 공개한다.

| 엔드포인트 | 설명 |
|---|---|
| `GET /health` | 상태 확인(인증 없음) |
| `POST /generate` | `{"messages": [...], "max_new_tokens": 1~300}` → `{"response": "...", "model": "..."}` (Bearer 키 필요) |

**실행 순서**: 런타임 유형을 **GPU(T4 이상)** 로 바꾼 뒤 위에서부터 차례로 실행한다.
**먼저 준비할 것**: 왼쪽 🔑 *보안 비밀*에 `NGROK_AUTHTOKEN`, `QWEN_API_KEY`를 등록한다(노트북 접근 허용 켜기). 키를 셀에 직접 쓰지 않는다.

- greedy decoding만 지원한다(temperature 미지원). JSON 출력·스트리밍·임베딩은 제공하지 않는다.
- 이 노트북이 켜져 있는 동안만 서버가 동작한다. 런타임이 끊기면 다시 실행한다.

In [ ]:
# 1) 패키지 설치 (torch / transformers 는 Colab 기본 설치본 사용)
!pip uninstall -y -q torchao  # Colab 기본 torchao가 최신 peft와 충돌하므로 제거 (이 노트북은 사용하지 않음)
!pip install -q fastapi "uvicorn[standard]" pyngrok peft accelerate

In [ ]:
# 2) 설정 — 여기만 수정한다
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"      # 베이스 모델 (LoRA 어댑터일 때 사용)
MODEL_DIR  = "/content/drive/MyDrive/CHANGE_ME/exp004"  # Drive 안의 EXP-004 폴더 (어댑터 또는 병합 모델)
MODEL_NAME = "Qwen2.5-3B-Instruct-EXP-004"   # /generate 응답의 model 값 (백엔드 기본값과 동일)
NGROK_DOMAIN = ""                            # 고정 도메인이 있으면 입력 예: "unfixed-shield-unsure.ngrok-free.dev" (없으면 비워 둠)
PORT = 8000

# 백엔드 QwenProvider 제한과 동일하게 유지
MAX_MESSAGES = 20
MAX_CHARS_PER_MESSAGE = 12000
MAX_NEW_TOKENS_LIMIT = 300

In [ ]:
# 3) Drive 마운트 + 비밀값 읽기
import os
from getpass import getpass
from google.colab import drive, userdata

drive.mount("/content/drive")

def get_secret(name):
    try:
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return getpass(f"{name} 입력(화면에 표시되지 않음): ")

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")
QWEN_API_KEY = get_secret("QWEN_API_KEY")
assert len(QWEN_API_KEY) >= 16, "QWEN_API_KEY는 16자 이상 임의 문자열을 권장합니다."
assert os.path.isdir(MODEL_DIR), f"MODEL_DIR을 찾을 수 없습니다: {MODEL_DIR}"
print("MODEL_DIR 내용:", sorted(os.listdir(MODEL_DIR))[:20])

In [ ]:
# 4) 모델 로드 — adapter_config.json 이 있으면 LoRA 로 보고 베이스에 얹고, 없으면 병합 모델로 로드
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 > 런타임 유형 변경에서 GPU를 선택하세요."

is_adapter = os.path.exists(os.path.join(MODEL_DIR, "adapter_config.json"))
# T4(compute capability 7.5)는 bf16을 소프트웨어로 흉내 내서 매우 느리다 → 8.0 이상(A100/L4 등)일 때만 bf16, 아니면 fp16
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

tok_src = MODEL_DIR if os.path.exists(os.path.join(MODEL_DIR, "tokenizer_config.json")) else (BASE_MODEL)
tokenizer = AutoTokenizer.from_pretrained(tok_src)

if is_adapter:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=dtype, device_map="auto")
    model = PeftModel.from_pretrained(base, MODEL_DIR)
    model = model.merge_and_unload()  # 어댑터를 가중치에 합쳐 추론 속도를 높인다
    print("LoRA 어댑터 로드:", MODEL_DIR)
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=dtype, device_map="auto")
    print("병합 모델 로드:", MODEL_DIR)
model.eval()
print("device:", next(model.parameters()).device, "dtype:", dtype)

In [ ]:
# 5) 생성 함수 + 간단 확인 (greedy)
import threading, time

_gpu_lock = threading.Lock()

def generate_text(messages, max_new_tokens=150):
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with _gpu_lock, torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

t0 = time.time()
print(generate_text([{"role": "user", "content": "안녕"}], max_new_tokens=40))
print(f"({time.time() - t0:.1f}s)")

In [ ]:
# 6) API 서버 (FastAPI) — 백엔드 QwenProvider 계약과 동일
import secrets
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="MINGLE Qwen EXP-004")

class Message(BaseModel):
    role: str
    content: str

class GenerateRequest(BaseModel):
    messages: list[Message]
    max_new_tokens: int = Field(default=150, ge=1, le=MAX_NEW_TOKENS_LIMIT)

def check_auth(authorization):
    expected = f"Bearer {QWEN_API_KEY}"
    if not authorization or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="unauthorized")

@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_NAME, "adapter": is_adapter}

@app.post("/generate")
def generate(req: GenerateRequest, authorization: str | None = Header(default=None)):
    check_auth(authorization)
    if not req.messages or len(req.messages) > MAX_MESSAGES:
        raise HTTPException(status_code=422, detail=f"messages must be 1..{MAX_MESSAGES}")
    if any(len(m.content) > MAX_CHARS_PER_MESSAGE for m in req.messages):
        raise HTTPException(status_code=422, detail=f"message too long (>{MAX_CHARS_PER_MESSAGE})")
    text = generate_text([m.model_dump() for m in req.messages], req.max_new_tokens)
    return {"response": text, "model": MODEL_NAME}

In [ ]:
# 7) 서버 시작 + ngrok 공개 (다시 실행하면 이전 서버/터널을 정리하고 재시작)
import uvicorn, requests
from pyngrok import ngrok

try:
    server.should_exit = True   # 이전 실행이 있으면 종료
    time.sleep(1)
except NameError:
    pass
ngrok.kill()

server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(30):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=1).ok:
            break
    except Exception:
        time.sleep(1)

ngrok.set_auth_token(NGROK_AUTHTOKEN)
kwargs = {"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}
tunnel = ngrok.connect(PORT, "http", **kwargs)
PUBLIC_URL = tunnel.public_url.replace("http://", "https://")

print("공개 URL :", PUBLIC_URL)
print("백엔드 backend/.env 에 아래를 설정하세요 (QWEN_API_KEY는 위 보안 비밀과 같은 값):")
print(f"  LLM_PROVIDER=qwen")
print(f"  QWEN_API_URL={PUBLIC_URL}")

In [ ]:
# 8) 외부 경로 자체 점검 (공개 URL → /health, /generate, 인증 실패)
H = {"Authorization": f"Bearer {QWEN_API_KEY}"}
r = requests.get(f"{PUBLIC_URL}/health", headers=H, timeout=20)
print("GET /health ->", r.status_code, r.json())

r = requests.post(f"{PUBLIC_URL}/generate", headers=H, timeout=120,
                  json={"messages": [{"role": "user", "content": "안녕"}], "max_new_tokens": 40})
print("POST /generate ->", r.status_code, r.json())

r = requests.post(f"{PUBLIC_URL}/generate", headers={"Authorization": "Bearer wrong"}, timeout=20,
                  json={"messages": [{"role": "user", "content": "x"}], "max_new_tokens": 5})
print("잘못된 키 ->", r.status_code, "(401이어야 정상)")

## 연결 확인 (로컬 맥에서)

`backend/.env`의 `LLM_PROVIDER`를 **한 줄만** `qwen`으로 두고, 위 7번 셀이 출력한 `QWEN_API_URL`과 같은 `QWEN_API_KEY`를 넣은 뒤:

```bash
cd backend && .venv/bin/python -m scripts.check_qwen
```

`POST /generate -> provider=qwen model=...`이 나오면 성공이다. 냉장고 회장님 대사만 Qwen으로 쓰려면 `LLM_PROVIDER=gemini`와 `FRIDGE_QWEN_DIALOGUE_ENABLED=true`를 사용한다.

In [ ]:
# 9) (선택) 시연이 끝나면 서버와 터널 종료
server.should_exit = True
ngrok.kill()
print("종료했습니다.")